# 05c Ship Calibrated

Ship the calibration artifacts written by `05b_calibrate_wshed.ipynb`. This notebook does not rescore Wflow against USGS; it only reads the reviewed calibration summary/patch and stamps downstream catalog artifacts.

## Runtime

In [ ]:
from pathlib import Path
import json

import pandas as pd
import yaml
from IPython.display import display

location_root = Path("..").resolve()

location_name = location_root.name
catalog_dir = location_root / "data/event_catalog/catalog"
scenario_catalog_path = catalog_dir / "scenario_catalog.csv"
calibration_root = location_root / "data/wflow/calibration"
calibration_summary_path = calibration_root / "usgs_wflow_calibration_summary.csv"
patch_json_path = calibration_root / "wflow_calibration_patch.json"
patch_yaml_path = calibration_root / "wflow_calibration_patch.yaml"

pd.set_option("display.max_colwidth", 160)
display(pd.Series({"location_root": str(location_root), "scenario_catalog": str(scenario_catalog_path)}))


## Rerun Control


In [ ]:
rerun = False


## Load 05b Calibration Artifacts

In [ ]:
if not calibration_summary_path.exists():
    raise FileNotFoundError(f"Run 05b_calibrate_wshed.ipynb first; missing {calibration_summary_path}")
if patch_json_path.exists():
    calibration_patch = json.loads(patch_json_path.read_text(encoding="utf-8"))
elif patch_yaml_path.exists():
    calibration_patch = yaml.safe_load(patch_yaml_path.read_text(encoding="utf-8")) or {}
else:
    raise FileNotFoundError(f"Run 05b_calibrate_wshed.ipynb first; missing {patch_json_path} or {patch_yaml_path}")

calibration_summary = pd.read_csv(calibration_summary_path, dtype={"event_id": str, "site_no": str})
display(pd.Series({
    "calibration_status": calibration_patch.get("status"),
    "primary_reference_gage": calibration_patch.get("primary_reference_gage"),
    "k_calibration": calibration_patch.get("k_calibration"),
    "event_count": calibration_patch.get("event_count"),
    "summary_rows": len(calibration_summary),
    "summary_csv": str(calibration_summary_path),
    "patch_json": str(patch_json_path) if patch_json_path.exists() else "not_written",
}, name="loaded_wflow_calibration"))
display(calibration_summary.head(20))


## Ship Catalog

In [ ]:
SHIP_CALIBRATED_CATALOG = True

scenario_catalog = pd.read_csv(scenario_catalog_path, dtype={"event_id": str})
calibrated_catalog = scenario_catalog.copy()
calibrated_catalog["wflow_calibration_status"] = calibration_patch.get("status")
calibrated_catalog["wflow_k_calibration"] = calibration_patch.get("k_calibration")
calibrated_catalog["wflow_calibration_reference_gage"] = calibration_patch.get("primary_reference_gage")
calibrated_catalog["wflow_calibration_patch"] = str(patch_yaml_path.relative_to(location_root))
calibrated_catalog["wflow_calibration_summary"] = str(calibration_summary_path.relative_to(location_root))

calibrated_catalog_path = catalog_dir / "scenario_catalog.calibrated.csv"
if SHIP_CALIBRATED_CATALOG:
    calibrated_catalog.to_csv(calibrated_catalog_path, index=False)

canonical_backup_path = catalog_dir / "scenario_catalog.pre_calibration_backup.csv"
if rerun:
    if not canonical_backup_path.exists():
        scenario_catalog.to_csv(canonical_backup_path, index=False)
    calibrated_catalog.to_csv(scenario_catalog_path, index=False)

display(pd.Series({
    "ship_calibrated_catalog": SHIP_CALIBRATED_CATALOG,
    "rerun": rerun,
    "calibrated_catalog": str(calibrated_catalog_path),
    "canonical_catalog": str(scenario_catalog_path),
    "canonical_backup": str(canonical_backup_path) if rerun else "not_written",
}, name="shipped_wflow_calibration"))
calibrated_catalog.head()
